# Module 2 — Python for AI Engineering

**Deep practice:** async I/O, bounded concurrency, retry taxonomy, cancellation, deadlines, cost accounting and production contracts.

Predict → Run → Observe → Break → Debug → Measure → Improve → Defend.


## Objectives + concept map
Concept map: async task → provider boundary → timeout → error classification → retry/backoff → concurrency limit → cancellation → cost/telemetry → contract.

Production rule: retries are a reliability mechanism, not a substitute for capacity planning or idempotency.


In [ ]:
import asyncio, time
from dataclasses import dataclass

@dataclass
class Response:
    text: str
    tokens: int

async def fake_llm(prompt, delay=.05, fail=False):
    await asyncio.sleep(delay)
    if fail: raise TimeoutError('provider timeout')
    return Response('answer: '+prompt[:30], len(prompt.split())+8)


## BUILD — retry contract
A retryable error should be classified explicitly. The loop must have a hard retry limit and exponential backoff. Predict the number of provider attempts before running.


In [ ]:
async def ask(prompt, retries=2):
    for attempt in range(retries+1):
        try: return await fake_llm(prompt)
        except TimeoutError:
            if attempt == retries: raise
            await asyncio.sleep(0.05 * 2**attempt)

print(asyncio.run(ask('Explain embeddings simply')))


## MEASURE — concurrency experiment
Compare 20 sequential/concurrent requests and concurrency limits 2, 5 and 10. Record wall time. Explain why unbounded concurrency can improve local latency while damaging provider reliability.


In [ ]:
async def batch(n=20, limit=None):
    sem = asyncio.Semaphore(limit) if limit else None
    async def one(i):
        if sem:
            async with sem: return await ask(f'question {i}')
        return await ask(f'question {i}')
    t=time.perf_counter(); out=await asyncio.gather(*(one(i) for i in range(n))); return time.perf_counter()-t,out
for limit in [1,2,5,10]: print('limit',limit,'seconds',round(asyncio.run(batch(20,limit))[0],3))


## TRY — cancellation and deadline
TODO: wrap a request in `asyncio.wait_for`, cancel a batch halfway through, and prove cleanup still happens. Then add a request ID and deadline to the response record.


## BREAK — failure injection
Inject: provider timeout; malformed response; cancellation; repeated transient failure; permanent 4xx-style failure; and retry storm. For each, classify **retry / do not retry / retry with escalation**.


In [ ]:
def classify(error):
    table = {'timeout':'retry','rate_limit':'retry_backoff','bad_request':'do_not_retry','auth':'do_not_retry','cancelled':'do_not_retry'}
    return table[error]
for e in ['timeout','rate_limit','bad_request','auth','cancelled']: print(e,'=>',classify(e))


## Industry scenario — enterprise training platform
10,000 learners may trigger model calls during a burst. Design limits for per-user concurrency, global concurrency, timeout, retry budget, provider fallback, and cost per task. Explain what happens when the provider is degraded for 10 minutes.


## Reference solution
Use bounded concurrency at both request and process/provider levels; enforce deadlines; retry only transient/rate-limit failures with exponential backoff and jitter; do not retry authorization/validation failures; propagate cancellation; cap retries; attach request/trace IDs; record tokens and estimated cost; use circuit breaking/fallback when degradation persists.


## Extension challenges
1. Add a token bucket rate limiter. 2. Add a circuit breaker. 3. Add idempotency keys. 4. Simulate provider 429s. 5. Compare fixed vs exponential backoff. 6. Add property tests for retry bounds. 7. Add streaming cancellation. 8. Produce p50/p95/p99 latency and cost/request.

### Mastery gate
Demonstrate bounded concurrency, correct retry classification, cancellation safety, hard deadlines and measurable cost. A solution that can retry forever fails the gate.
